# QCNN - transpilation analysis

The hardware-oriented analysis evaluates the practical feasibility of the proposed QCNN architectures by examining how their logical circuits are transformed when **transpiled** onto realistic quantum hardware, together with the number of trainable parameters associated with each architecture as an indicator of trainability.


## 1. Setup

Import libraries, locate the project, and configure the analysis settings and table display.


In [ ]:
# import libraries for data loading, circuit conversion and transpilation
from __future__ import annotations

from pathlib import Path
import sys

import hashlib
import json
import time

import numpy as np
import pandas as pd
import pennylane as qml
import torch
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import fake_provider

# import the notebook table display helper
from IPython.display import display

# find the project root directory by looking for the training script
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "training" / "train.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the notebook from a directory inside the qcnn project.")

# make the repository packages available to the notebook
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from datasets import get_dataset_config, get_test_loader
from models import get_model, list_models
from models.utilities.encoding import (
    compute_e1_gamma, e1_global_stats_4, e1_quad_means_8, patches_4x15,
)
from models.utilities.qcnn_model import OUTPUT_WIRES


In [ ]:
# Compare the selected trained models on one shared dataset and training seed.
MODELS = list_models()
DATASET = "fashion4_1"  # mnist, organamnist, fashion4_1, fashion4_2 or fashion4_3
TRAINING_SEED = 42

# use local backend snapshots with identical compilation settings for every model
BACKEND_NAMES = ("FakeBerlin", "FakeSherbrooke", "FakeTorino")
OPTIMIZATION_LEVEL = 3
TRANSPILER_SEED = 42  # independent of the training seed

# select one reproducible test image
IMAGE_SEED = 42

# Restore trained weights and use one official test image. The input-dependent
# encoding features are recomputed for that image before circuit transpilation.
RESULTS_DIR = PROJECT_ROOT / "results"
DATA_DIR = PROJECT_ROOT / "datasets" / "data"


In [ ]:
# configure table display in the notebook
pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 24)


## 2. Circuit Parameter Retrieval

Load the optimized weights for the selected model/dataset/seed runs and recompute image-dependent encoding values using one common official test image. This reconstructs the complete trained circuit for that test input, ready for transpilation. Missing dataset files are downloaded by the shared dataset loaders.


In [ ]:
# compute file hashes without modifying the source files
def file_sha256(path: Path) -> str:
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


# read training metadata from existing JSON files
def read_json(path: Path) -> dict:
    with Path(path).open(encoding="utf-8") as stream:
        return json.load(stream)


# validate the requested runs and collect their best-checkpoint metadata
def select_best_runs(results_dir, model_names, dataset_name, seed):
    """Load each model's best epoch from one explicitly selected dataset and seed."""
    results_dir = Path(results_dir)
    metadata = get_dataset_config(dataset_name)
    candidates, problems = [], []
    # require complete training files and matching run metadata
    for model in model_names:
        run_dir = results_dir / model / dataset_name / f"seed{seed}"
        required = ("config.json", "test.json", "best_state.pt")
        missing = [name for name in required if not (run_dir / name).is_file()]
        if missing:
            problems.append(f"{run_dir}: missing {', '.join(missing)}")
            continue
        config, result = read_json(run_dir / "config.json"), read_json(run_dir / "test.json")
        if any(
            record.get("model") != model or record.get("dataset") != dataset_name
            or record.get("seed") != seed
            or any(record.get(key) != value for key, value in metadata.items())
            for record in (config, result)
        ):
            problems.append(f"{run_dir}: inconsistent model/dataset/seed/classes")
            continue
        accuracy, loss = float(result["val_acc_best"]), float(result["best_val_loss"])
        if not np.isfinite([accuracy, loss, result["test_acc"]]).all():
            problems.append(f"{run_dir}: non-finite metrics")
            continue
        candidates.append({
            "model": model, "dataset": dataset_name, "selected_seed": seed,
            "val_acc": accuracy, "val_loss": loss,
            "test_acc": float(result["test_acc"]),
            "best_epoch": int(result["best_epoch"]),
            "topology": config["topology"], "ansatz": config["ansatz"],
            "encoding": config["encoding"], "trainable_params": int(result["n_params"]),
            "checkpoint": str((run_dir / "best_state.pt").resolve()),
        })
    if problems:
        raise FileNotFoundError(
            "Complete all requested training runs before analysis.\n" + "\n".join(problems)
        )
    if not candidates:
        raise ValueError("Select at least one model.")
    # record checkpoint hashes to detect changes before circuit construction
    best = pd.DataFrame(candidates)
    best["checkpoint_sha256"] = best["checkpoint"].map(lambda path: file_sha256(Path(path)))
    return best.reset_index(drop=True)


# restore trained weights and check checkpoint and architecture consistency
def load_best_model(row):
    path = Path(row["checkpoint"])
    if file_sha256(path) != row["checkpoint_sha256"]:
        raise RuntimeError(f"Checkpoint changed after selection: {path}")
    # check saved metrics before restoring the trained parameters
    saved = torch.load(path, map_location="cpu", weights_only=False)
    if (
        saved["seed"] != row["selected_seed"]
        or saved["epoch"] != row["best_epoch"]
        or not np.isclose(saved["val_acc"], row["val_acc"], rtol=0, atol=1e-10)
        or not np.isclose(saved["val_loss"], row["val_loss"], rtol=0, atol=1e-10)
    ):
        raise RuntimeError(f"Best checkpoint disagrees with test.json: {path}")
    # Constructor placeholders are always overwritten; no random-weight fallback.
    with torch.random.fork_rng(devices=[]):
        model = get_model(row["model"])
    model.load_state_dict(saved["model_state"], strict=True)
    # verify that the current architecture matches the training configuration
    config = model.model_config()
    for key in ("topology", "ansatz", "encoding"):
        if config[key] != row[key]:
            raise RuntimeError(f"Architecture changed since training: {path} ({key})")
    if model.n_trainable_params() != row["trainable_params"]:
        raise RuntimeError(f"Trainable parameter count changed: {path}")
    return model.eval().requires_grad_(False)


In [ ]:
# select one trained checkpoint per model and display its metrics
selected = select_best_runs(RESULTS_DIR, MODELS, DATASET, TRAINING_SEED)
print(f"{len(selected)} models: {DATASET}, training seed {TRAINING_SEED}.")


In [ ]:
# Select one common test image through the same dataset adapters as training.
def pick_common_image(data_dir, dataset_name, image_seed=42):
    """One reproducible image, preprocessed for each encoder's input grid."""
    metadata = get_dataset_config(dataset_name)
    views = {}
    for size in (28, 8, 16):
        loader = get_test_loader(
            dataset_name, data_dir=data_dir, image_size=size, seed=image_seed,
            batch_size=1, test_per_class=1,
        )
        # Seeded selection is independent of image_size: every view uses the
        # same source image. The first batch contains one of the four classes.
        pixels, target = next(iter(loader))
        views[size] = pixels
        if size == 28:
            output_label = int(target[0])
            index = int(loader.dataset.indices[0])
    return views, {
        "source": f"{dataset_name} official test split", "dataset": dataset_name,
        "index": index, "original_label": metadata["original_labels"][output_label],
        "class_name": metadata["class_names"][output_label],
        "image_seed": int(image_seed),
        "sha256": hashlib.sha256(views[28].numpy().tobytes()).hexdigest(),
    }


def image_tensor(image, size):
    """Return the view already produced by the selected training adapter."""
    return image[size].reshape(1, -1)


In [ ]:
# select the common input image and keep its metadata in memory
image, image_info = pick_common_image(DATA_DIR, DATASET, IMAGE_SEED)


## 3. Trained circuits and logical resources

Convert the complete trained circuits, including the input encoding, to Qiskit via QASM in memory, and display logical resource counts.


In [ ]:
# trace the trained model with its actual input feature extraction
def model_tape(model, x):
    """Trace the repository QNode, including the real input feature extraction."""
    # reproduce the input features and parameter order of each encoding
    tail = (model.theta_fine, model.theta_pool1, model.theta_coarse, model.theta_pool2)
    if model.encoding_name == "reupload":
        args = (patches_4x15(x), model.theta_reupload, model.weight_reupload, *tail)
    elif model.encoding_name == "e1":
        args = (
            e1_quad_means_8(x), compute_e1_gamma(e1_global_stats_4(x)),
            model.a_embed, model.c_embed, *tail,
        )
    elif model.encoding_name == "e3":
        args = (x, *tail)
    else:
        raise ValueError(f"Unsupported encoding: {model.encoding_name}")
    # remove broadcasting and convert traced parameters to NumPy
    tape = model.qnode.construct(args, {})
    tapes, _ = qml.transforms.broadcast_expand(tape)
    if len(tapes) != 1:
        raise ValueError("Analysis requires exactly one input image.")
    tapes, _ = qml.transforms.convert_to_numpy_parameters(tapes[0])
    return tapes[0]


# collect logical or native circuit resource counts
def circuit_metrics(circuit):
    # exclude measurements and barriers from gate counts and circuit depth
    gates = [inst for inst in circuit.data if inst.operation.name not in {"measure", "barrier"}]
    used = {circuit.find_bit(qubit).index for inst in gates for qubit in inst.qubits}
    ops = {str(name): int(count) for name, count in circuit.count_ops().items()}
    return {
        "depth": int(circuit.depth(filter_function=lambda inst: inst.operation.name not in {"measure", "barrier"}) or 0),
        "size_with_measure": int(circuit.size()),
        "gate_count": len(gates),
        "oneq_count": sum(inst.operation.num_qubits == 1 for inst in gates),
        "twoq_count": sum(inst.operation.num_qubits == 2 for inst in gates),
        "twoq_depth": int(circuit.depth(filter_function=lambda inst: inst.operation.name not in {"measure", "barrier"} and inst.operation.num_qubits == 2) or 0),
        "multiq_count": sum(inst.operation.num_qubits > 2 for inst in gates),
        "physical_qubits_used": len(used), "ops": json.dumps(ops, sort_keys=True),
    }


# convert the trained circuit to Qiskit
def export_model_circuit(model, image):
    x = image_tensor(image, model.image_size)
    with torch.no_grad():
        tape = model_tape(model, x)
        # convert the complete trained circuit without writing a QASM file
        qasm = qml.to_openqasm(
            tape, wires=range(model.n_qubits), measure_all=False, rotations=False,
        )
        circuit = QuantumCircuit.from_qasm_str(qasm)
        if circuit_metrics(circuit)["multiq_count"]:
            raise RuntimeError("QASM baseline must decompose all gates to one/two-qubit gates.")
        # validate the output-wire mapping
        measurements = [inst for inst in circuit.data if inst.operation.name == "measure"]
        measured_wires = [circuit.find_bit(inst.qubits[0]).index for inst in measurements]
        if measured_wires != list(OUTPUT_WIRES) or circuit.num_clbits != len(OUTPUT_WIRES):
            raise RuntimeError("Unexpected output measurement mapping.")
    return circuit, qasm, {
        "logical_qubits": model.n_qubits,
        "template_operations": len(tape.operations),
        "template_twoq_count": sum(len(op.wires) == 2 for op in tape.operations),
        "template_multiq_count": sum(len(op.wires) > 2 for op in tape.operations),
    }


In [ ]:
# construct circuits and return their metrics without creating analysis files
def build_circuits(selected_runs, image):
    circuits, rows = {}, []
    for index, (_, row) in enumerate(selected_runs.iterrows(), start=1):
        model = load_best_model(row)
        circuit, _, extra = export_model_circuit(model, image)
        key = (row["model"], row["dataset"])
        circuits[key] = circuit
        # retain the circuit and its resource metrics in memory
        metrics = {f"raw_{name}": value for name, value in circuit_metrics(circuit).items()}
        rows.append({**row.to_dict(), **extra, **metrics})
        print(f"[{index}/{len(selected_runs)}] {key[0]} / {key[1]}: loaded seed {row['selected_seed']}, converted", flush=True)
    logical = pd.DataFrame(rows)
    return circuits, logical


In [ ]:
# build the selected trained circuits and display their logical resource counts
circuits, logical = build_circuits(selected, image)
display(logical[["model", "trainable_params",
                 "logical_qubits", "raw_gate_count", "raw_depth", "raw_twoq_count",
                 "raw_twoq_depth"]].round(7))


## 4. Hardware Transpilation

Inspect local backend snapshots and transpile each circuit using the configured optimization level and a single transpiler seed. Report native gate counts, circuit depth and compilation overhead for each model and backend.


In [ ]:
# instantiate the requested local backend snapshots
def make_backends(names):
    backends = {}
    for name in names:
        cls = getattr(fake_provider, name, None)
        if cls is None:
            raise ImportError(f"{name} is unavailable in the installed qiskit-ibm-runtime package.")
        backends[name] = cls()
    return backends


# collect backend metadata and a snapshot hash in memory
def backend_manifest(backends):
    rows = []
    for name, backend in backends.items():
        properties = backend.properties()
        snapshot = {"configuration": backend.configuration().to_dict(), "properties": properties.to_dict()}
        rows.append({
            "backend": name, "qubits": backend.num_qubits,
            "basis_gates": sorted(backend.operation_names),
            "last_update_date": str(properties.last_update_date),
            "snapshot_sha256": hashlib.sha256(json.dumps(snapshot, sort_keys=True, default=str).encode()).hexdigest(),
        })
    return rows


In [ ]:
# create the backend objects and display their hardware metadata
backends = make_backends(BACKEND_NAMES)
display(pd.DataFrame(backend_manifest(backends))[["backend", "qubits", "basis_gates", "last_update_date"]])


In [ ]:
# compile each model once per backend using one transpiler seed
def transpile_comparison(circuits, logical, backends, optimization_level=3, transpiler_seed=42):
    if not backends:
        raise ValueError("Select at least one backend.")
    if not isinstance(transpiler_seed, (int, np.integer)):
        raise ValueError("The transpiler seed must be a single integer.")
    rows = []
    indexed = logical.set_index(["model", "dataset"])
    total = len(circuits) * len(backends)
    for backend_name, backend in backends.items():
        # apply the same target and pass-manager settings to every model
        pm = generate_preset_pass_manager(
            backend=backend, optimization_level=optimization_level,
            seed_transpiler=int(transpiler_seed), approximation_degree=1.0,
        )
        for (model_name, dataset), circuit in circuits.items():
            start = time.perf_counter()
            compiled = pm.run(circuit)
            elapsed = time.perf_counter() - start
            # check that every compiled instruction is supported by the target
            for inst in compiled.data:
                qargs = tuple(compiled.find_bit(qubit).index for qubit in inst.qubits)
                if not backend.target.instruction_supported(operation_name=inst.operation.name, qargs=qargs):
                    raise RuntimeError(f"Unsupported instruction on {backend_name}: {inst.operation.name} {qargs}")
            # compare native resource counts with the logical QASM baseline
            source = indexed.loc[(model_name, dataset)].to_dict()
            metrics = circuit_metrics(compiled)
            rows.append({
                "model": model_name, "dataset": dataset, **source, **metrics,
                "backend": backend_name, "transpiler_seed": int(transpiler_seed),
                "optimization_level": optimization_level, "transpile_seconds": elapsed,
                "depth_expansion": metrics["depth"] / source["raw_depth"] if source["raw_depth"] else np.nan,
            })
            # report progress while retaining intermediate results in memory
            print(f"[{len(rows)}/{total}] {backend_name} / {model_name} / {dataset}: 2Q={metrics['twoq_count']}, depth={metrics['depth']} ({elapsed:.1f}s)", flush=True)
    return pd.DataFrame(rows)


In [ ]:
# run compilation with the shared settings and keep the resulting table in memory
transpiled = transpile_comparison(
    circuits, logical, backends,
    optimization_level=OPTIMIZATION_LEVEL, transpiler_seed=TRANSPILER_SEED,
)


In [ ]:
# report the single compilation result for each model and backend
summary = transpiled[[
    "backend", "dataset", "model", "topology", "ansatz", "encoding", "transpiler_seed",
    "twoq_count", "twoq_depth", "depth", "gate_count",
    "physical_qubits_used",
]].copy()

# display native resource counts without exporting a CSV file
display(summary.round(2))


In [ ]:
# compute the RZ share of one-qubit gates for each model on FakeBerlin
berlin_rz = transpiled.loc[
    transpiled["backend"].eq("FakeBerlin"),
    ["model", "dataset", "oneq_count", "ops"],
].copy()
if berlin_rz.empty:
    raise ValueError("No FakeBerlin results: run the transpilation cells first.")
if berlin_rz["oneq_count"].eq(0).any():
    raise ValueError("The RZ percentage is undefined for models with no one-qubit gates.")

berlin_rz["rz_count"] = berlin_rz["ops"].map(lambda ops: json.loads(ops).get("rz", 0))
berlin_rz["rz_percentage"] = 100 * berlin_rz["rz_count"] / berlin_rz["oneq_count"]
display(berlin_rz[["model", "dataset", "rz_count", "oneq_count", "rz_percentage"]].round(2))

# arithmetic mean of the per-model percentages, with equal weight for each model
mean_rz_percentage = berlin_rz["rz_percentage"].mean()
print(f"FakeBerlin: percentuale media di RZ sui gate a 1 qubit ({len(berlin_rz)} modelli): {mean_rz_percentage:.2f}%")
